In [4]:
from arcgis.gis import GIS
from arcgis.features import analysis
from pathlib import Path
import arcpy
import shutil

In [5]:
my_gis_org = GIS("home")
my_gis_org.users.me


<User username:dave_crawford@esri.com_prof_services>

In [6]:
source_fgdb_path = Path(r".\Parking_Violations_and_Neighborhoods.gdb")

In [7]:
arcpy.management.Compact(str(source_fgdb_path))

<Result 'Parking_Violations_and_Neighborhoods.gdb'>

In [8]:
# The name of the folder to place the File Geodatabase in
fgdb_folder_name = source_fgdb_path.stem
# The location of the folder to place the File Geodatabase in
fgdb_folder_location = source_fgdb_path.parent
# The path to the folder to place the File Geodatabase in
fgdb_folder_path = fgdb_folder_location.joinpath(fgdb_folder_name)
# The path of our copied File Geodatabase
fgdb_path = fgdb_folder_path.joinpath(source_fgdb_path.name)

In [10]:
# Copy the File Geodatabase to the new location
shutil.copytree(source_fgdb_path, fgdb_path)
# Zip the File Geodatabase
zipped_fgdb = shutil.make_archive(
    base_name=fgdb_folder_path,  # The name of the archive, not including the file extension
    format="zip",  # The archive format
    root_dir=fgdb_folder_path,  # The directory to archive
) 

In [11]:
fgdb_item_properties = {
    "type": "File Geodatabase",  # The type of item this will be
    "title": "Philadelphia Parking Tickets by Neighborhood",  # The title of the item
    "description": "Parking tickets issued in Philadelphia by neighborhood",  # The description of the item
    "tags": "Philadelphia, Parking, Tickets, Neighborhood",  # Tags for the item
}


In [12]:
my_fgdb_item = my_gis_org.content.add(
    item_properties=fgdb_item_properties,  # The properties of the item
    data=zipped_fgdb,  # The path to the zipped file geodatabase
)

c:\Users\dav11274\AppData\Local\ESRI\conda\envs\quacker34\Lib\site-packages\IPython\core\interactiveshell.py:3550: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
my_fgdb_item

<Item title:"Philadelphia Parking Tickets by Neighborhood" type:File Geodatabase owner:dave_crawford@esri.com_prof_services>

In [14]:
fgdb_publish_parameters = {
    "name": "Philadelphia Parking Tickets by Neighborhood",  # The name of the service
} 

In [15]:
my_hosted_feature_service_item = my_fgdb_item.publish(
    publish_parameters=fgdb_publish_parameters
)

In [16]:
my_hosted_feature_service_item

<Item title:"Philadelphia Parking Tickets by Neighborhood" type:Feature Layer Collection owner:dave_crawford@esri.com_prof_services>

In [17]:
[layer.properties["name"] for layer in my_hosted_feature_service_item.layers]

['Philadelphia_Neighborhoods', 'Parking_Violations_Dec_2017']

In [18]:
parking_tickets_by_neighborhood_layer = my_hosted_feature_service_item.layers[0]

In [19]:
enriched_layer = analysis.enrich_layer(
    input_layer=parking_tickets_by_neighborhood_layer,  # FeatureLayer object to be enriched
    analysis_variables=[
        "crime.CRMCYPROC"
    ],  # Analysis variables to enrich the input layer with
    output_name="Philadelphia Parking Tickets by Neighborhood Enriched with Property Crime",  # The name of the output layer
)

c:\Users\dav11274\AppData\Local\ESRI\conda\envs\quacker34\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'analysis1.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
{"cost": -1}


'99e38076e5dc4be3889b237edaff9c4a'